# Apache Spark — Core Concepts & Optimization

Training notebook covering Spark internals: distributed processing, lazy evaluation, execution model, performance monitoring (Serverless tools and the classic-compute Spark UI & logs), transformations, streaming, and optimization techniques.

## 1. Distributed Processing Architecture

### How Spark Distributes Work

```
┌─────────────────────────────────────────────────────────┐
│                     DRIVER PROGRAM                      |
│  • Schedules tasks                • Collects results    │
└──────────────────────┬──────────────────────────────────┘
                       │  distributes tasks
       ┌───────────────┼───────────────┐
       ▼               ▼               ▼
 ┌───────────┐   ┌───────────┐   ┌───────────┐
 │ EXECUTOR 1│   │ EXECUTOR 2│   │ EXECUTOR 3│
 │ ┌───┬───┐ │   │ ┌───┬───┐ │   │ ┌───┬───┐ │
 │ │T1 │T2 │ │   │ │T3 │T4 │ │   │ │T5 │T6 │ │
 │ └───┴───┘ │   │ └───┴───┘ │   │ └───┴───┘ │
 └───────────┘   └───────────┘   └───────────┘
    Worker 1        Worker 2        Worker 3
```

### Key Concepts

| Component | Role |
|---|---|
| **Driver** | Orchestrates the application, creates DAG, assigns tasks |
| **Executor** | JVM process on worker node, runs tasks & stores data |
| **Task** | Smallest unit of work — operates on **one partition** |
| **Partition** | Chunk of data processed in parallel |
| **Cluster Manager** | Allocates resources (managed by Databricks on Serverless) |

> **Key insight:** More partitions = more parallelism (up to available cores)

In [0]:
from pyspark.sql.functions import spark_partition_id, count

# --- Distributed Processing in Action ---

# Create a DataFrame and observe partitioning
df = spark.range(1_000_000)

# Count partitions using spark_partition_id() (serverless-compatible)
part_count = df.select(spark_partition_id().alias("pid")).distinct().count()
print(f"Number of partitions: {part_count}")

# Repartition and verify
df_repart = df.repartition(10)
part_count_8 = df_repart.select(spark_partition_id().alias("pid")).distinct().count()
print(f"After repartition(8): {part_count_8}")

# Coalesce (reduce partitions WITHOUT full shuffle)
df_coal = df_repart.coalesce(2)
part_count_2 = df_coal.select(spark_partition_id().alias("pid")).distinct().count()
print(f"After coalesce(2)   : {part_count_2}")

# Show data distribution across partitions
print("\n--- Rows per partition (repartitioned to 8) ---")
display(
    df_repart
    .groupBy(spark_partition_id().alias("partition_id"))
    .agg(count("*").alias("row_count"))
    .orderBy("partition_id")
)

In [0]:
from pyspark.sql.functions import col, sum as _sum

# --- Lazy: Nothing executes yet ---
orders = spark.read.table("samples.tpch.orders")



In [0]:

# Chain of transformations — all lazy
result = (
    orders
    .filter(col("o_orderstatus") == "F")
    .groupBy("o_orderpriority")
    .agg(_sum("o_totalprice").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)

print("✅ No computation happened yet — only a plan was built")
print("─" * 55)

# View the execution plan
result.explain(mode="formatted")

In [0]:
# --- Trigger execution with an ACTION ---
# This creates: 1 Job → multiple Stages → Tasks
# Check the Spark UI after running!

display(result)  # ACTION: triggers the full DAG


### Transformations vs Actions

| Transformations (Lazy) | Actions (Trigger Execution) |
|---|---|
| `select`, `filter`, `withColumn` | `count`, `show`, `collect` |
| `groupBy`, `join`, `orderBy` | `write`, `save`, `display` |
| `map`, `flatMap`, `union` | `first`, `take`, `foreach` |
| Build a **logical plan** — nothing runs yet | **Trigger** the DAG → jobs → stages → tasks |

### Execution Hierarchy

```
Action (e.g. df.count())
 └── Job  (one per action)
      ├── Stage 1  (separated by shuffles)
      │    ├── Task 1  (one per partition)
      │    └── Task 2
      └── Stage 2
           ├── Task 3
           └── Task 4
```

### Execution Plans

| Plan | What It Shows |
|---|---|
| **Parsed Logical Plan** | Raw user intent |
| **Analyzed Logical Plan** | Resolved columns & types |
| **Optimized Logical Plan** | Catalyst optimizer rewrites |
| **Physical Plan** | Actual execution strategy (scans, joins, exchanges) |

> **Key insight:** Spark optimizes the **entire chain** of transformations before running anything

## 3. Performance Monitoring on Serverless

> ⚠️ **Serverless does NOT have the traditional Spark UI.** Use these tools instead. On **classic compute**, the Spark UI and driver/executor logs are available — see **Section 3b**.

### How to Monitor on Serverless

| Tool | What It Shows | How to Access |
|---|---|---|
| **`explain()`** | Physical plan — scans, joins, shuffles, filters | `df.explain()` or `df.explain(mode="formatted")` |
| **`explain("cost")`** | Plan + estimated row counts & sizes | `df.explain(mode="cost")` |
| **Query Profile** | Visual execution plan in SQL Editor | Run query in SQL Editor → click **"Query Profile"** |
| **Query History** | Past queries, duration, rows scanned | Left sidebar → **Query History** |

### Reading an Execution Plan

```
AdaptiveSparkPlan          ← AQE wrapper (always present)
 +- PhotonSort              ← orderBy()
    +- Exchange              ← SHUFFLE (wide transformation!)
       +- PhotonGroupingAgg  ← groupBy().agg()
          +- PhotonScan      ← table/file scan (filters pushed here)
```

### What to Look For in `explain()`

| Look For | Meaning |
|---|---|
| **PhotonScan** with `DataFilters` | ✅ Predicate pushdown is working |
| **Exchange** | ⚠️ Shuffle happening — wide transformation |
| **PhotonBroadcastHashJoin** | ✅ Small table broadcast (no shuffle) |
| **SortMergeJoin** | ⚠️ Both sides shuffled — expensive |
| **AdaptiveSparkPlan** | ✅ AQE is active |
| **ReadSchema** with fewer columns | ✅ Column pruning is working |

> 💡 **Tip:** For visual execution plans, run your query in the **SQL Editor** and click **"Query Profile"**

In [0]:
from pyspark.sql.functions import col, count, avg

# --- On Serverless: use explain() to inspect execution plans ---
lineitem = spark.read.table("samples.tpch.lineitem")

heavy_query = (
    lineitem
    .filter(col("l_shipdate") >= "1995-01-01")
    .groupBy("l_returnflag", "l_linestatus")
    .agg(
        count("*").alias("count_orders"),
        avg("l_quantity").alias("avg_qty"),
        avg("l_discount").alias("avg_discount")
    )
    .orderBy("l_returnflag", "l_linestatus")
)

# 1️⃣ Formatted plan — shows each operator with details
print("=== FORMATTED PLAN ===")
heavy_query.explain(mode="formatted")

print("\n" + "═" * 55)

# 2️⃣ Cost plan — includes estimated row counts
print("\n=== COST PLAN (with row estimates) ===")
heavy_query.explain(mode="cost")

# 3️⃣ Execute and display results
print("\n=== RESULTS ===")
display(heavy_query)

print("\n💡 On Serverless: explain() is your primary debugging tool")
print("   For visual plans: run the SQL in the SQL Editor → Query Profile")

## 3b. Spark UI & Logs on Classic Compute

> Applies to **classic all-purpose / job clusters** (Dedicated or No-isolation access mode). Serverless and Standard (shared) compute do not expose the Spark UI — use Section 3 instead.

### How to Open the Spark UI

| From | Path |
|---|---|
| **Notebook cell** | Expand **"Spark Jobs"** under a cell's output → click **View** next to a job |
| **Cluster page** | **Compute** → select cluster → **Spark UI** tab |
| **Job run** | **Jobs & Pipelines** → run → task → **Spark UI** link |
| **Terminated cluster** | Spark UI stays available (event-log based) for a limited time after termination |

### Spark UI Tabs

| Tab | What It Shows | Use It To |
|---|---|---|
| **Jobs** | One row per action; duration, stages, status | Find the slow action |
| **Stages** | Tasks per stage, shuffle read/write, spill, task time distribution (min/median/max) | Detect **skew** (max >> median) and **spill** |
| **Storage** | Cached DataFrames/tables, memory vs disk usage | Verify caching worked / free memory |
| **Environment** | Spark configs, JVM & classpath | Confirm `spark.sql.*` settings |
| **Executors** | Cores, memory, GC time, failed tasks, **stdout/stderr per executor** | Spot GC pressure, lost executors, executor logs |
| **SQL / DataFrame** | Visual physical plan with **runtime metrics** per operator (rows, bytes, time) | See actual rows per Exchange/Join, AQE final plan |
| **Structured Streaming** | Input rate, process rate, batch duration per stream | Check whether a stream keeps up |

### Execution Hierarchy in the UI

```
Jobs tab      →  Job 12  (display(result))
Stages tab    →    ├── Stage 30  Scan + partial agg  (200 tasks)   shuffle write
              →    └── Stage 31  Final agg + sort    (AQE: 4 tasks) shuffle read
SQL tab       →  Query 7: visual DAG with rows/bytes per operator
```

### Red Flags to Look For

| Symptom (where) | Likely Cause | Fix |
|---|---|---|
| Max task time >> median (Stages → Summary Metrics) | **Data skew** | AQE skew join, salting, filter hot keys |
| **Spill (Memory/Disk)** > 0 (Stages) | Partitions too large for executor memory | More shuffle partitions, larger nodes |
| Large **Shuffle Read/Write** (Stages) | Wide transformations, big joins | Broadcast small tables, filter/prune earlier |
| High **GC Time** (Executors) | Memory pressure, large objects | Bigger memory, fewer cached objects |
| Thousands of tiny tasks (Stages) | Too many small files/partitions | `OPTIMIZE`, coalesce, AQE coalescing |
| Failed/killed executors (Executors) | OOM, spot instance loss | Check executor stderr / driver log4j |

### Driver & Executor Logs

| Log | Where | Contains |
|---|---|---|
| **Driver `stdout`** | Compute → cluster → **Driver logs** | `print()` output from driver-side code |
| **Driver `stderr`** | Compute → cluster → **Driver logs** | Python tracebacks, warnings |
| **Driver `log4j`** | Compute → cluster → **Driver logs** | Spark/JVM internals: scheduling, AQE decisions, OOM errors |
| **Executor `stdout`/`stderr`** | Spark UI → **Executors** tab → per-executor links | Output/errors from code running on workers (e.g. UDFs) |
| **Event log** | Compute → cluster → **Event log** | Cluster lifecycle: start, resize, autoscaling, terminations |
| **Cluster log delivery** | Cluster config → **Advanced → Logging** (UC Volume or DBFS path) | Persists driver/executor/init-script logs after termination |

> 💡 **Tip:** Label your jobs with `sc.setJobDescription()` (next cell) so they are easy to find in the Jobs and SQL tabs.

In [0]:
from pyspark.sql.functions import col, sum as _sum, count

# --- Classic compute only: SparkContext gives access to Spark UI labelling ---
# On Serverless / Standard (Spark Connect) spark.sparkContext is not available.
try:
    sc = spark.sparkContext
    is_classic = True
except Exception:
    is_classic = False

print(f"spark.sql.shuffle.partitions = {spark.conf.get('spark.sql.shuffle.partitions')}")

if not is_classic:
    print("\n⚠️ Serverless / Standard compute detected — no Spark UI.")
    print("   Attach a classic cluster (Dedicated access mode) to run this example,")
    print("   or use explain() / Query Profile from Section 3.")
else:
    print(f"Spark version       : {sc.version}")
    print(f"Application ID      : {sc.applicationId}")
    print(f"Default parallelism : {sc.defaultParallelism}  (total executor cores)")

    # Group related jobs and give each a readable name in the Jobs / SQL tabs
    sc.setJobGroup("dox_training", "Spark training - TPCH queries")

    orders = spark.read.table("samples.tpch.orders")
    lineitem = spark.read.table("samples.tpch.lineitem")

    # Job 1: narrow + wide (filter -> groupBy -> orderBy)
    sc.setJobDescription("01 - Revenue by priority (filter + groupBy + orderBy)")
    revenue = (
        orders.filter(col("o_orderstatus") == "F")
        .groupBy("o_orderpriority")
        .agg(_sum("o_totalprice").alias("total_revenue"))
        .orderBy(col("total_revenue").desc())
    )
    display(revenue)

    # Job 2: join -> look for SortMergeJoin vs BroadcastHashJoin in the SQL tab
    sc.setJobDescription("02 - Lineitems per order status (join + groupBy)")
    joined = (
        lineitem.join(orders, lineitem.l_orderkey == orders.o_orderkey)
        .groupBy("o_orderstatus")
        .agg(count("*").alias("line_count"))
    )
    display(joined)

    # Reset labels so later cells are not mislabelled
    sc.setJobDescription(None)
    sc.setJobGroup(None, None)

    print("""
💡 Now open the Spark UI (expand 'Spark Jobs' above -> View):
   • Jobs tab   : find jobs '01 - ...' and '02 - ...'
   • Stages tab : compare shuffle read/write and task time min/median/max
   • SQL tab    : open the query -> see rows per operator, Exchange nodes, join strategy
   • Executors  : GC time and per-executor stdout/stderr
""")

In [0]:
from pyspark.sql.functions import col, when, count

# --- Create an intentionally skewed key and aggregate on it ---
# ~90% of rows land on key 0 -> one shuffle partition gets most of the data.
skewed = (
    spark.range(5_000_000)
    .withColumn("key", when(col("id") % 10 < 9, 0).otherwise(col("id") % 1000))
)

try:
    spark.sparkContext.setJobDescription("03 - Skewed groupBy (check Stages -> Summary Metrics)")
except Exception:
    pass  # Serverless / Standard: no Spark UI labels

skew_result = (
    skewed.groupBy("key")
    .agg(count("*").alias("rows"))
    .orderBy(col("rows").desc())
)
display(skew_result.limit(10))

print("""💡 In the Spark UI (classic):
   Stages tab -> open the shuffle-read stage -> 'Summary Metrics for Tasks'
   • Duration / Shuffle Read Size: Max is far above the Median => SKEW
   • Check 'Spill (Memory)' / 'Spill (Disk)' columns for spilling tasks
   SQL tab -> AQE may show 'AQEShuffleRead' with coalesced/skew-split partitions
""")

## 4. Narrow vs Wide Transformations

### Comparison

| | Narrow | Wide |
|---|---|---|
| **Data movement** | None — each partition works independently | **Shuffle** — data moves across nodes |
| **Stage boundary** | Same stage | Creates a **new stage** |
| **Examples** | `select`, `filter`, `map`, `withColumn`, `union` | `groupBy`, `join`, `orderBy`, `distinct`, `repartition` |
| **Performance** | ✅ Fast, pipelined | ⚠️ Expensive — network + disk I/O |

### Visual

```
NARROW (no shuffle)              WIDE (shuffle = stage boundary)

 Partition 1 → Partition 1       Partition 1 ─┬─→ Partition A
 Partition 2 → Partition 2       Partition 2 ─┼─→ Partition B
 Partition 3 → Partition 3       Partition 3 ─┴─→ Partition C
                                  ▲ data redistributed
```

> **Key insight:** Every wide transformation = a shuffle = network I/O = **the #1 performance bottleneck in Spark**

In [0]:
from pyspark.sql.functions import col, upper

orders = spark.read.table("samples.tpch.orders")

# --- NARROW transformations (no shuffle, same stage) ---
narrow_result = (
    orders
    .select("o_orderkey", "o_orderstatus", "o_totalprice")
    .filter(col("o_totalprice") > 1000)
    .withColumn("status_upper", upper(col("o_orderstatus")))
)

print("=== NARROW Plan (no Exchange/shuffle) ===")
narrow_result.explain()

print("\n" + "═" * 55)

# --- WIDE transformations (shuffle required) ---
wide_result = (
    orders
    .groupBy("o_orderstatus")
    .count()
    .orderBy(col("count").desc())
)

print("\n=== WIDE Plan (Exchange = shuffle!) ===")
wide_result.explain()

## 5. Batch vs Streaming Workloads

### Comparison

| | Batch | Streaming |
|---|---|---|
| **Data** | Bounded, finite dataset | Continuous, unbounded data |
| **Processing** | Start → process all → end | Always running, processes as data arrives |
| **Latency** | Minutes to hours | Seconds to minutes |
| **API** | `spark.read` → `df.write` | `spark.readStream` → `df.writeStream` |
| **Use Cases** | ETL, reports, backfills | Real-time dashboards, alerts, CDC |

### Structured Streaming — Unified API

```
 Batch:     spark.read.table("src")  →  transform  →  df.write.table("tgt")
 Stream:    spark.readStream("src")  →  transform  →  df.writeStream("tgt")
                                        ▲
                                  same DataFrame API!
```

### Trigger Modes

| Trigger | Behavior |
|---|---|
| `availableNow=True` | Process all available data, then stop (like batch but incremental) |
| `processingTime='10 seconds'` | Micro-batch every 10 seconds |
| `once=True` | Process one micro-batch, then stop |
| *(default)* | Next micro-batch as soon as previous finishes |

In [0]:
from pyspark.sql.functions import col, current_timestamp

# ========== BATCH ==========
# Read all data at once → transform → display
batch_df = (
    spark.read.table("samples.tpch.orders")
    .filter(col("o_orderstatus") == "O")
    .select("o_orderkey", "o_totalprice", "o_orderdate")
    .limit(5)
)
print("=== BATCH: spark.read → bounded, finite ===")
display(batch_df)

# ========== STREAMING PATTERN ==========
# Streaming uses the SAME DataFrame API, just swap read → readStream
print("\n=== STREAMING: spark.readStream → unbounded, continuous ===")

# Build the streaming DataFrame (lazy — no data flows yet)
stream_df = (
    spark.readStream
    .format("rate")                   # synthetic source: generates rows/sec
    .option("rowsPerSecond", 10)
    .load()
    .withColumn("processed_at", current_timestamp())
)

# Print the schema — same DataFrame API as batch
print("Streaming DataFrame schema:")
stream_df.printSchema()

print("""\n💡 To run a stream on Serverless, checkpoint must be on a UC Volume:

   display(stream_df, checkpointLocation="/Volumes/training/bronze/checkpoints/my_stream")

   OR write to a table:

   stream_df.writeStream \
       .format("delta") \
       .option("checkpointLocation", "/Volumes/training/bronze/checkpoints/my_stream") \
       .trigger(processingTime="10 seconds") \
       .toTable("training.bronze.target_table")
""")

In [0]:
# --- Managing active streaming queries ---
# List all active streams
for q in spark.streams.active:
    print(f"Active stream: name={q.name}, id={q.id}, status={q.status}")

if not spark.streams.active:
    print("No active streams (expected — we only showed the pattern above)")

print("""\n💡 Common stream management commands:
   spark.streams.active           → list running streams
   query.stop()                   → stop a specific stream
   query.awaitTermination()       → block until stream ends
   query.lastProgress             → latest micro-batch metrics
""")

## 6. Spark Optimization

### Optimization Techniques at a Glance

| Technique | What It Does | When to Use |
|---|---|---|
| **Predicate Pushdown** | Pushes filters to data source — reads less data | Always — automatic with Spark |
| **Column Pruning** | Reads only selected columns | Use `select()` instead of `select("*")` |
| **AQE** | Dynamically adjusts at runtime | Enabled by default in Databricks |
| **Partition Tuning** | Right-sizes parallelism | Skew, too many/few tasks |

### Adaptive Query Execution (AQE)

AQE optimizes **at runtime** based on actual data statistics:

| AQE Feature | What It Does |
|---|---|
| **Coalesce Shuffle Partitions** | Merges small partitions → fewer tasks |
| **Skew Join Optimization** | Splits skewed partitions into sub-partitions |

> 💡 AQE is **enabled by default** in Databricks — look for `AdaptiveSparkPlan` in explain()

In [0]:
from pyspark.sql.functions import col

orders = spark.read.table("samples.tpch.orders")

# --- BAD: Reads all columns, filters late ---
bad = orders.select("*").filter(col("o_orderstatus") == "F")

# --- GOOD: Select only needed columns, filter early ---
good = orders.select("o_orderkey", "o_totalprice", "o_orderstatus").filter(col("o_orderstatus") == "F")

print("=== BAD: select(*) then filter ===")
bad.explain()

print("\n=== GOOD: select + filter (pushdown) ===")
good.explain()

print("\n💡 Notice: Spark pushes the filter INTO the scan in both cases")
print("   But selecting fewer columns = less data read from disk")

In [0]:
from pyspark.sql.functions import col, sum as _sum

# --- AQE: Adaptive Query Execution ---
# AQE is always ON in Databricks (serverless & classic)
print("✅ AQE is enabled by default in Databricks")
print("   • Coalesces small shuffle partitions")
print("   • Converts sort-merge to broadcast joins at runtime")
print("   • Handles skewed join partitions")

# Run a query with shuffle — AQE will optimize partition count
orders = spark.read.table("samples.tpch.orders")

agg_result = (
    orders
    .groupBy("o_orderstatus", "o_orderpriority")
    .agg(_sum("o_totalprice").alias("total"))
    .orderBy(col("total").desc())
)

# Check the plan: look for "AdaptiveSparkPlan"
print("\n=== Plan with AQE ===")
agg_result.explain(mode="formatted")

print("\n💡 Look for 'AdaptiveSparkPlan' at the top of the plan")
print("   After execution it shows 'isFinalPlan=true' with optimized stages")

display(agg_result)